# Diagnose failures and observe behavior

Use controlled HTTP responses to see how the real RAM client and Agent Framework providers handle missing
sessions, unavailable recall, load/write policy, and cancellation. Capture connector activities and metrics in
memory using `ActivityListener` and `MeterListener`; no telemetry collector or model is needed.

Build the solution in Release and start a fresh C# kernel in `notebooks/csharp`. These are offline transport
experiments, not a live service health check. No secrets, `.env` file, Redis deployment, or OpenAI key is required.
Every experiment disposes its HTTP client and listeners within the cell so reruns do not accumulate listeners.

The provider inspection hooks are experimental in the pinned Agent Framework 1.19.0 API; these cells explicitly suppress its `MAAI001` diagnostic.


In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
#nullable enable
#r "nuget: Microsoft.Agents.AI, 1.19.0"
#r "nuget: Microsoft.Extensions.DependencyInjection, 10.0.12"
#r "nuget: ModelContextProtocol, 2.2.0"
#r "../../src/Redis.Iris.AgentMemory/bin/Release/net8.0/Redis.Iris.AgentMemory.dll"
#r "../../src/Redis.Iris.ContextRetriever.Modeling/bin/Release/net8.0/Redis.Iris.ContextRetriever.Modeling.dll"
#r "../../src/Redis.Iris.ContextRetriever/bin/Release/net8.0/Redis.Iris.ContextRetriever.dll"
#r "../../src/Redis.Iris.AgentFramework/bin/Release/net8.0/Redis.Iris.AgentFramework.dll"

using System;
using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Text.Json;
using System.Text.Json.Serialization;
using System.Threading;
using System.Threading.Tasks;
using Microsoft.Agents.AI;
using Redis.Iris.AgentFramework;
using Microsoft.Extensions.AI;
using Microsoft.Extensions.DependencyInjection;
using Redis.Iris.AgentMemory;
using Redis.Iris.ContextRetriever;
using Redis.Iris.ContextRetriever.Modeling;

using System.Net;
using System.Net.Http;
using System.Diagnostics;

In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
#load "memory-lab.csx"

## A missing session is an expected history miss

`TryGetSessionMemoryAsync` maps HTTP 404 to no history. Its trace has `outcome=not_found` and unset status; it does
not increment the error counter. A strict `GetSessionMemoryAsync` call still throws and records an error.
The transport below returns 404; this is not caused by a bad credential or real service.

In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
using (var telemetry = new MemoryLabTelemetry())
using (var http = new HttpClient(new MemoryLabTransport { MissingHistory = true }))
{
    var client = MemoryLab.Client(http);
    var history = new RedisIrisChatHistoryProvider(client, _ => MemoryLab.State());
    var messages = (await history.InvokingAsync(new(MemoryLab.Agent, await MemoryLab.Agent.CreateSessionAsync(), Array.Empty<ChatMessage>()))).ToList();
    MemoryLab.Check(messages.Count == 0, "Missing history should be empty.");
    var miss = telemetry.Spans.Single();
    MemoryLab.Check(miss.Tags["redis.iris.outcome"]?.ToString() == "not_found" && miss.Status == ActivityStatusCode.Unset, "Expected a non-error miss.");
    MemoryLab.Check(!telemetry.Measurements.Any(item => item.Name == "redis.iris.errors"), "Expected no error metric for history miss.");
    Console.WriteLine($"Provider history: {messages.Count} messages; trace outcome={miss.Tags["redis.iris.outcome"]}; status={miss.Status}");
    bool strictFailed = false;
    try { await client.GetSessionMemoryAsync("session-avery"); }
    catch (AgentMemoryException error) when (error.StatusCode == HttpStatusCode.NotFound) { strictFailed = true; }
    MemoryLab.Check(strictFailed && telemetry.Measurements.Where(item => item.Name == "redis.iris.errors").Sum(item => item.Value) == 1, "Strict read should record one error.");
    Console.WriteLine("Strict get: throws HTTP 404 and increments errors once.");
}

## Unavailable recall: fail open or fail closed

Both modes make one request and record an HTTP 503 failure. Fail-open continues with the original prompt and no
recalled message. Fail-closed propagates the error so the host can stop the invocation. A successful agent response
would not prove recall succeeded. The optional logger is deliberately omitted; this notebook displays only safe
status/type summaries, never raw exception bodies.

In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
foreach (var failClosed in new[] { false, true })
{
    var transport = new MemoryLabTransport { FailureStatus = HttpStatusCode.ServiceUnavailable };
    using var http = new HttpClient(transport);
    using var telemetry = new MemoryLabTelemetry();
    var provider = new RedisIrisMemoryProvider(new RedisIrisLongTermMemorySearch(MemoryLab.Client(http)), _ => MemoryLab.State(), new() { FailClosed = failClosed });
    var failed = false;
    var delivered = -1;
    try
    {
        var messages = (await provider.InvokingAsync(new(MemoryLab.Agent, await MemoryLab.Agent.CreateSessionAsync(),
            new[] { MemoryLab.Question(MemoryLab.SeatingQuestion) }))).ToList();
        delivered = messages.Count;
        MemoryLab.Check(delivered == 1 && messages[0].Text == MemoryLab.SeatingQuestion, "Fail-open should retain only the prompt.");
    }
    catch (AgentMemoryException error) when (error.StatusCode == HttpStatusCode.ServiceUnavailable) { failed = true; }
    MemoryLab.Check(failed == failClosed && transport.Sends == 1, "Unexpected failure behavior or retry.");
    MemoryLab.Check(telemetry.Spans.Single().Status == ActivityStatusCode.Error, "HTTP failure should remain visible in telemetry.");
    Console.WriteLine($"Recall failClosed={failClosed}: {(failed ? "invocation stopped" : "prompt retained; no recall")}; HTTP sends={transport.Sends}; error spans=1");
}

## History load and write policies are separate

History load defaults to fail-closed; recall defaults to fail-open; event writes default to fail-closed. Explicitly
compare both policies for loads and writes below. The write experiment fails before any data is stored in the
fixture. In a real service, a failed response or cancellation can leave an unknown write outcome: do not retry
blindly, and do not describe fail-open writes as persisted history.

In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
foreach (var policy in new[] { RedisIrisFailurePolicy.FailOpen, RedisIrisFailurePolicy.FailClosed })
{
    foreach (var operation in new[] { "load", "write" })
    {
        var transport = new MemoryLabTransport { FailureStatus = HttpStatusCode.ServiceUnavailable };
        using var http = new HttpClient(transport);
        var history = new RedisIrisChatHistoryProvider(MemoryLab.Client(http), _ => MemoryLab.State(), options: new()
        { LoadFailurePolicy = policy, WriteFailurePolicy = policy });
        var session = await MemoryLab.Agent.CreateSessionAsync();
        bool failed = false;
        try
        {
            if (operation == "load")
            {
                var loaded = await history.InvokingAsync(new(MemoryLab.Agent, session, Array.Empty<ChatMessage>()));
                MemoryLab.Check(!loaded.Any(), "Failed load should provide no history.");
            }
            else await history.InvokedAsync(new(MemoryLab.Agent, session,
                new[] { MemoryLab.Question("Synthetic notebook event.") }, Array.Empty<ChatMessage>()));
        }
        catch (AgentMemoryException error) when (error.StatusCode == HttpStatusCode.ServiceUnavailable) { failed = true; }
        MemoryLab.Check(failed == (policy == RedisIrisFailurePolicy.FailClosed) && transport.Sends == 1, "Unexpected history policy behavior.");
        Console.WriteLine($"History {operation} / {policy}: {(failed ? "stopped" : "continued without successful storage operation")}; one HTTP send.");
    }
}

## Cancellation is never converted into successful fail-open recall

Wait until the request reaches the fixture, then cancel it. The explicit cancellation is deterministic and avoids
a timing race from a tiny timeout. The provider must propagate `OperationCanceledException` even with
`FailClosed = false`. The real client records the canceled operation as an error; it does not retry it.

In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
var blocked = new MemoryLabTransport { BlockUntilCancelled = true };
using (var http = new HttpClient(blocked))
using (var telemetry = new MemoryLabTelemetry())
using (var cancellation = new CancellationTokenSource())
{
    var recall = new RedisIrisMemoryProvider(new RedisIrisLongTermMemorySearch(MemoryLab.Client(http)), _ => MemoryLab.State(), new() { FailClosed = false });
    var pending = recall.InvokingAsync(new(MemoryLab.Agent, await MemoryLab.Agent.CreateSessionAsync(),
        new[] { MemoryLab.Question(MemoryLab.SeatingQuestion) }), cancellation.Token).AsTask();
    bool canceled = false;
    try
    {
        await blocked.Entered.Task.WaitAsync(TimeSpan.FromSeconds(5));
        cancellation.Cancel();
        try { await pending.WaitAsync(TimeSpan.FromSeconds(5)); }
        catch (OperationCanceledException) { canceled = true; }
    }
    finally { cancellation.Cancel(); }
    MemoryLab.Check(canceled && blocked.Sends == 1, "Cancellation must propagate without retry.");
    MemoryLab.Check(telemetry.Spans.Single().Status == ActivityStatusCode.Error, "Canceled operation should be recorded.");
    Console.WriteLine("Cancellation propagated; no empty-success fallback and no retry.");
}

## Inspect safe diagnostic dimensions

Observe one successful recall. These are connector-generated tags, not broad HTTP instrumentation or MCP SDK
payload logging. Metrics have bounded operation/outcome dimensions; they do not include owners, sessions,
queries, memory IDs, or text. Durations vary across runs. Assertions inspect both tag names and sensitive synthetic
values to keep the example honest. The displayed context in notebook 07 is separate from telemetry.

In [ ]:
#pragma warning disable MAAI001 // Inspect the experimental Agent Framework provider hooks.
using (var telemetry = new MemoryLabTelemetry())
using (var http = new HttpClient(new MemoryLabTransport()))
{
    var provider = new RedisIrisMemoryProvider(new RedisIrisLongTermMemorySearch(MemoryLab.Client(http)), _ => MemoryLab.State(),
        new() { FailClosed = true, Search = new() { Limit = 2, SimilarityThreshold = .70 } });
    await provider.InvokingAsync(new(MemoryLab.Agent, await MemoryLab.Agent.CreateSessionAsync(), new[] { MemoryLab.Question(MemoryLab.SeatingQuestion) }));
    var allowedSpanTags = new[] { "redis.iris.operation", "server.address", "redis.iris.retry_count", "redis.iris.outcome", "http.response.status_code", "redis.iris.result_count", "error.type" };
    var allowedMetricTags = new[] { "redis.iris.operation", "redis.iris.outcome", "redis.iris.reason" };
    foreach (var span in telemetry.Spans)
    {
        MemoryLab.Check(span.Tags.Keys.All(allowedSpanTags.Contains), "Unexpected trace dimension.");
        Console.WriteLine($"Span {span.Name}: {span.Status}; {JsonSerializer.Serialize(span.Tags)}");
    }
    foreach (var measurement in telemetry.Measurements)
    {
        MemoryLab.Check(measurement.Tags.Keys.All(allowedMetricTags.Contains), "Unexpected metric dimension.");
        Console.WriteLine($"Metric {measurement.Name}={measurement.Value:F2}; {JsonSerializer.Serialize(measurement.Tags)}");
    }
    var serialized = JsonSerializer.Serialize(new { Spans = telemetry.Spans.ToArray(), Metrics = telemetry.Measurements.ToArray() });
    foreach (var forbidden in new[] { "avery", "session-avery", "avery-seat", "synthetic-notebook-key", MemoryLab.SeatingQuestion, "window seat" })
        MemoryLab.Check(!serialized.Contains(forbidden, StringComparison.OrdinalIgnoreCase), "Sensitive synthetic value appeared in diagnostics.");
    MemoryLab.Check(telemetry.Measurements.Any(item => item.Name == "redis.iris.recall.results" && item.Value == 2), "Expected result-count measurement.");
    Console.WriteLine("Diagnostic dimensions verified. All clients and listeners are disposed at cell exit.");
}

## Use this evidence when troubleshooting

| Symptom | Check next |
| --- | --- |
| Empty history with `not_found` | New/missing session; distinguish from strict read failures |
| Empty recall with successful HTTP | Owner/topic filters, threshold, promotion/indexing lag (notebook 07) |
| Agent continued after a recall error | Fail-open policy and connector error metrics |
| 401 / 403 | Correct service credential, delegated token expiry/issuer, host and service permissions |
| Cancellation | Caller deadline or shutdown; it does not mean successful empty recall |
| Missing durable history after fail-open write | Storage failure; reconcile before any retry |

See [observability](../../OBSERVABILITY.md) for collection and [Context Retriever diagnostics](../../docs/context-retriever.md)
for MCP connection, discovery, tool-call timeouts, and cache health. This notebook validates RAM behavior only;
Context Retriever emits a separate source/meter. Do not turn on sensitive SDK payload logging to troubleshoot
production prompts or customer data. No service state was changed by these exercises.